# IDX Financial Extraction Database Analysis
This notebook demonstrates how to query the DuckDB medallion warehouse (`data/idxfin.duckdb`) generated by the extraction pipeline.

It covers:
1. **Bronze Layer**: Document ingestion metadata and hash integrity
2. **Reference Layer**: Taxonomy concept catalog
3. **Gold Layer**: Multi-issuer financial summaries across Mining (NCKL), Infrastructure (IPCM), and Banking (BBSI)
4. **Lineage & Provenance**: Tracing reported financial figures back to exact PDF source pages
5. **Silver Facts**: Deep-dive into granular normalized line items
6. **Corporate Governance**: Board of Directors and Commissioners
7. **Audit Intelligence**: Independent auditor report, signing partners, opinions, and Key Audit Matters (KAMs)
8. **Forward Guidance**: Management targets, capex plans, and operational outlook
9. **Validation Engine**: Accounting equation audit trail (zero diff checks)

In [18]:
import duckdb
import pandas as pd

# Connect to DuckDB warehouse (read-only mode is supported)
conn = duckdb.connect('data/idxfin.duckdb', read_only=True)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print("Connected to DuckDB successfully!")

Connected to DuckDB successfully!


In [31]:
conn.close()

## 1. Bronze Layer: Ingested Documents
View all filings ingested into the Bronze store with SHA-256 provenance hashes, currency, and detected scales.

In [2]:
df_docs = conn.execute("""
    SELECT doc_id[:12] AS doc_id_short, issuer_id, doc_type, fiscal_year, fiscal_period, 
           period_end, currency, scale, page_count, ingested_at
    FROM idxfin_bronze.document
    ORDER BY issuer_id, doc_type
""").df()
df_docs

,doc_id_short,issuer_id,doc_type,fiscal_year,fiscal_period,period_end,currency,scale,page_count,ingested_at
0,34e92be251a7,BBSI,annual_report,2025,FY,2025-12-31,IDR,1000000,453,2026-09-10 20:16:23.060928
1,5c582cdc9d2f,BBSI,fs_annual,2025,FY,2025-12-31,IDR,1,68,2026-09-10 20:14:31.792960
2,adf1eae6656d,IPCM,annual_report,2025,FY,2025-12-31,IDR,1000000,592,2026-09-10 20:16:08.276628
3,20401326f76b,IPCM,fs_annual,2025,FY,2025-12-31,IDR,1000,79,2026-09-10 20:13:36.858090
4,c55de1d3c362,NCKL,annual_report,2025,FY,2025-12-31,IDR,1000000,416,2026-09-10 20:01:46.677208
5,51cb2d29aa08,NCKL,fs_annual,2025,FY,2025-12-31,IDR,1000000,98,2026-09-10 20:12:22.574487


## 2. Gold Layer: Multi-Industry Financial Summary
Standardized financial metrics across all extracted issuers. Notice how industry-specific presentations are handled:
- **General / Mining (NCKL)**: Reports full Gross Profit and Operating Cash Flow.
- **Infrastructure / Port Services (IPCM)**: Reports Service Revenue and Gross Profit.
- **Commercial Banking (BBSI / Bank Krom)**: Reports Net Interest Income, Bank Loans, and Customer Deposits (`unavailable` for Gross Profit/COGS as per banking IFRS standards).

In [7]:
df_summary = conn.execute("""
    SELECT *
    FROM idxfin_gold.financial_summary
    ORDER BY issuer_id, period_end DESC
""").df()

# Format numbers with commas, displaying 'unavailable' where applicable
formatted = df_summary.copy()
num_cols = [c for c in df_summary.columns if c not in ('issuer_id', 'period_end', 'fiscal_year', 'fiscal_period')]
for c in num_cols:
    formatted[c] = formatted[c].apply(lambda x: f"{x:,.0f}" if pd.notnull(x) else "unavailable")
formatted

,issuer_id,period_end,fiscal_year,fiscal_period,total_assets,current_assets,total_liabilities,current_liabilities,total_equity,revenue,cost_of_revenue,gross_profit,net_profit,operating_cash_flow,investing_cash_flow,financing_cash_flow,cash_and_equivalents,capex,interest_income,net_interest_income,bank_loans,bank_deposits
0,BBSI,2025-12-31,2025,FY,"12,214,396,484,644",unavailable,"8,752,048,122,301",unavailable,"3,462,348,362,343","2,280,328,111,081","427,719,101,021","1,852,609,010,060","143,284,784,067","175,161,642,051","-71,854,717,189","-2,777,584,214","729,234,678,033","19,935,500,750","2,280,328,111,081","1,852,609,010,060","8,332,054,844,101","8,398,428,898,005"
1,BBSI,2024-12-31,2024,FY,"6,651,447,899,693",unavailable,"3,331,408,004,480",unavailable,"3,320,039,895,213","1,083,828,965,262","118,765,517,776","965,063,447,486","124,060,349,375","502,690,672,567","-33,122,968,215","-481,950,000","628,705,337,385","1,976,027,436","1,083,828,965,262","965,063,447,486","3,975,559,833,358","3,159,827,983,495"
2,IPCM,2025-12-31,2025,FY,"1,714,517,440,000","1,015,317,877,000","354,723,262,000","350,802,317,000","1,359,794,178,000","1,474,807,574,000","1,045,017,043,000","429,790,531,000","196,437,892,000","355,728,037,000","-29,710,476,000","-244,975,358,000","554,126,337,000","29,710,476,000","17,405,669,000",unavailable,unavailable,unavailable
3,IPCM,2024-12-31,2024,FY,"1,649,981,015,000","938,837,610,000","360,870,838,000","354,135,181,000","1,289,110,177,000","1,345,030,257,000","926,892,911,000","418,137,346,000","166,842,883,000","329,456,805,000","-26,147,743,000","-239,041,305,000","473,084,134,000","26,147,743,000","16,168,375,000",unavailable,unavailable,unavailable
4,NCKL,2025-12-31,2025,FY,"61,770,103,000,000","14,159,490,000,000","15,014,238,000,000","6,657,018,000,000","46,755,865,000,000","29,632,799,000,000","19,941,401,000,000","9,691,398,000,000","10,970,337,000,000","8,601,085,000,000","-4,791,345,000,000","-4,504,237,000,000","6,016,371,000,000","331,318,000,000","260,235,000,000",unavailable,unavailable,unavailable
5,NCKL,2024-12-31,2024,FY,"52,253,838,000,000","13,464,563,000,000","15,799,784,000,000","5,853,566,000,000","36,454,054,000,000","26,965,262,000,000","18,517,976,000,000","8,447,286,000,000","7,712,368,000,000","5,708,382,000,000","-2,271,537,000,000","-1,086,900,000,000","6,486,402,000,000","657,104,000,000","246,415,000,000",unavailable,unavailable,unavailable


In [25]:
tmp_df = conn\
    .execute("""
    SELECT issuer_id, fiscal_year, concept_id, label_raw, raw_text, unit_id, value
FROM idxfin_silver.fact
WHERE concept_id LIKE 'op.%' and issuer_id = 'AMMN'
ORDER BY issuer_id, fiscal_year;
    
    """).df()

In [19]:
# 1. Total facts count
count = conn.execute("SELECT COUNT(*) FROM idxfin_silver.fact WHERE issuer_id = 'AMMN'").fetchone()[0]
print(f"Total AMMN facts in silver.fact: {count}")  # -> 1156
# 2. View AMMN's revenue, COGS, and profit in the Gold layer
print(conn.execute("""
    SELECT issuer_id, fiscal_year, fiscal_period, currency, revenue, cost_of_revenue, gross_profit, net_profit
    FROM idxfin_gold.financial_summary
    WHERE issuer_id = 'AMMN'
    ORDER BY fiscal_year
""").df())


Total AMMN facts in silver.fact: 1156
  issuer_id  fiscal_year fiscal_period currency       revenue  cost_of_revenue  gross_profit    net_profit
0      AMMN         2022            FY      USD  2.830122e+09     1.191250e+09  1.638872e+09  1.098772e+09
1      AMMN         2023            FY      USD  2.033365e+09     1.131441e+09  9.019240e+08  2.588890e+08
2      AMMN         2024            FY      USD  2.663630e+09     1.318596e+09  1.345034e+09  6.416770e+08
3      AMMN         2025            FY      USD  1.846521e+09     1.010898e+09  8.356230e+08  2.580380e+08


In [26]:
tmp_df.sample(10).T

,1,2,8,4,6,0,9,7,5,3
issuer_id,AMMN,AMMN,AMMN,AMMN,AMMN,AMMN,AMMN,AMMN,AMMN,AMMN
fiscal_year,2023,2023,2024,2024,2024,2023,2025,2024,2024,2023
concept_id,op.core.production_volume,op.core.production_volume,op.core.production_volume,op.core.production_volume,op.core.installed_capacity,op.core.production_volume,op.core.installed_capacity,op.core.production_volume,op.core.unit_cash_cost,op.core.unit_cash_cost
label_raw,Copper production,concentrate production,Copper production,gold production,kapasitas input,gold production,kapasitas input,produksi konsentrat,Adjusted C1 cash cost,Adjusted C1 cash cost
raw_text,312,"541,894",395,802.749,40,463.467,85,755.083,"(3,37)","(0,73)"
unit_id,million pounds,dry metric tons,million pounds,ounces,juta ton per tahun,ounces,juta ton per tahun,metrik ton kering,US$/pon terjual,US$/pon terjual
value,312.0,541894.0,395.0,802.749,40.0,463.467,85.0,755.083,-337.0,-73.0


In [30]:
df_breakdown = conn.execute("""
    SELECT 
        issuer_id,
        lineage_extractor,
        CASE WHEN concept_id LIKE 'fs.%' THEN 'Financial Statement'
             WHEN concept_id LIKE 'op.%' THEN 'Operational KPI'
             ELSE 'Extension / Disclosure' END AS category,
        COUNT(*) AS fact_count
    FROM idxfin_silver.fact
    where concept_id like 'op.%'
    GROUP BY issuer_id,lineage_extractor, category
    ORDER BY fact_count DESC
""").df()
df_breakdown

,issuer_id,lineage_extractor,category,fact_count
0,AALI,ar.mdna.kpis,Operational KPI,211
1,ANJT,ar.mdna.kpis,Operational KPI,45
2,ADRO,ar.mdna.kpis,Operational KPI,42
3,ALDO,ar.mdna.kpis,Operational KPI,32
4,ADMG,ar.mdna.kpis,Operational KPI,21
5,ANDI,ar.mdna.kpis,Operational KPI,18
6,AMMN,ar.mdna.kpis,Operational KPI,10
7,AKRA,ar.mdna.kpis,Operational KPI,9
8,AGII,ar.mdna.kpis,Operational KPI,4
9,NCKL,ar.mdna.kpis,Operational KPI,3


In [27]:
tmp_df.shape

(10, 7)

## 3. Metric Provenance & Grounding Citations
Every metric in the Gold summary traces directly to the source page in the original PDF filing.

In [ ]:
df_prov = conn.execute("""
    SELECT issuer_id, period_end, fiscal_year,
           total_assets_page, total_liabilities_page, total_equity_page,
           revenue_page, net_profit_page, operating_cash_flow_page
    FROM idxfin_gold.financial_summary_provenance
    ORDER BY issuer_id, period_end DESC
""").df()
df_prov.fillna('-')

## 4. Silver Layer: Granular Normalized Facts
Query line items extracted from primary statements and note disclosures.

In [ ]:
df_facts = conn.execute("""
    SELECT issuer_id, concept_id, label_raw, period_end, value, raw_text, scale_as_reported, lineage_page
    FROM idxfin_silver.fact
    WHERE concept_id IN ('fs.is.revenue', 'fs.is.profit_for_the_period', 'fs.bs.total_assets', 'fs.cf.net_cash_operating')
    ORDER BY issuer_id, concept_id, period_end DESC
""").df()
df_facts

## 5. Corporate Governance: Board Profiles
Board of Directors and Board of Commissioners extracted from Annual Reports.

In [ ]:
df_officers = conn.execute("""
    SELECT issuer_id, person_name, role, is_independent, source_page
    FROM idxfin_silver.officer_position
    ORDER BY issuer_id, role, person_name
""").df()
df_officers

## 6. Audit Intelligence: Independent Auditor Reports
Auditor names, audit network affiliation, signing public accountants, opinions, and Key Audit Matters (KAMs).

In [ ]:
df_audit = conn.execute("""
    SELECT issuer_id, fiscal_year, audit_firm, signing_partner, opinion, key_audit_matters, source_page
    FROM idxfin_silver.audit_report
""").df()
df_audit

## 7. Management Forward Guidance & Capex Outlook
Forward-looking statements, growth targets, and capex budgets extracted from the MD&A Outlook.

In [ ]:
df_guidance = conn.execute("""
    SELECT issuer_id, metric_label_raw, scenario, point_raw, low_raw, high_raw, unit_raw, target_period_raw, source_page
    FROM idxfin_silver.forward_guidance
""").df()
df_guidance

## 8. Accounting Validation Engine Audit Trail
Audit trail showing accounting identity verification results (Balance Sheet identities, Gross Profit, Net Profit) down to zero difference.

In [ ]:
df_val = conn.execute("""
    SELECT rule_id, passed, CAST(lhs AS BIGINT) AS lhs, CAST(rhs AS BIGINT) AS rhs, 
           CAST(diff AS BIGINT) AS diff, message, checked_at
    FROM idxfin_silver.validation_result
    WHERE passed = true
    ORDER BY checked_at DESC
    LIMIT 15
""").df()
df_val